# 🎬 Movie Recommendation Agent

A simple multi-tool AI agent using **GPT-4o-mini + LangChain + LangGraph + LangSmith**.

### Workflow
```text
User → Agent → Tool → Agent → Final Answer
              ↑
        ┌─────┼─────┐
        │     │     │
     Search Details Top Rated
```


In [ ]:
!pip install -qU langchain langchain-openai langgraph langsmith

In [ ]:
import os
from typing import Annotated, TypedDict

from google.colab import userdata
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

print("Imports successful!")

In [ ]:
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
OPENAI_BASE_URL = "https://keygateway1.arshnivlabs.com/v1"
OPENAI_MODEL = "gpt-4o-mini"

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in Colab Secrets.")

llm = ChatOpenAI(
    model=OPENAI_MODEL,
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL,
    temperature=0
)

print("LLM initialized!")
print("Model:", OPENAI_MODEL)

In [ ]:
response = llm.invoke("What is a movie recommendation system?")
print(response.content)

## 1. Movie Dataset

In [ ]:
MOVIES = [
    {"title": "Inception", "genre": "Sci-Fi", "rating": 8.8, "year": 2010,
     "director": "Christopher Nolan",
     "description": "A skilled thief enters people's dreams to steal and plant ideas."},
    {"title": "The Dark Knight", "genre": "Action", "rating": 9.0, "year": 2008,
     "director": "Christopher Nolan",
     "description": "Batman faces the Joker, a criminal mastermind who creates chaos in Gotham."},
    {"title": "Interstellar", "genre": "Sci-Fi", "rating": 8.7, "year": 2014,
     "director": "Christopher Nolan",
     "description": "Explorers travel through a wormhole searching for a new home for humanity."},
    {"title": "The Matrix", "genre": "Sci-Fi", "rating": 8.7, "year": 1999,
     "director": "The Wachowskis",
     "description": "A hacker discovers that reality is actually a simulated world."},
    {"title": "John Wick", "genre": "Action", "rating": 7.4, "year": 2014,
     "director": "Chad Stahelski",
     "description": "A retired assassin returns to the criminal underworld seeking revenge."},
    {"title": "Mad Max: Fury Road", "genre": "Action", "rating": 8.1, "year": 2015,
     "director": "George Miller",
     "description": "A high-speed chase across a post-apocalyptic wasteland."},
    {"title": "Avengers: Endgame", "genre": "Action", "rating": 8.4, "year": 2019,
     "director": "Anthony Russo, Joe Russo",
     "description": "The Avengers attempt to reverse the destruction caused by Thanos."},
    {"title": "The Shawshank Redemption", "genre": "Drama", "rating": 9.3, "year": 1994,
     "director": "Frank Darabont",
     "description": "Two prisoners develop a friendship while surviving life in prison."},
    {"title": "Forrest Gump", "genre": "Drama", "rating": 8.8, "year": 1994,
     "director": "Robert Zemeckis",
     "description": "A kind-hearted man experiences several major moments in American history."},
    {"title": "The Hangover", "genre": "Comedy", "rating": 7.7, "year": 2009,
     "director": "Todd Phillips",
     "description": "Three friends try to reconstruct what happened after a wild bachelor party."},
    {"title": "Superbad", "genre": "Comedy", "rating": 7.6, "year": 2007,
     "director": "Greg Mottola",
     "description": "Two high-school friends attempt to make the most of their final days before graduation."},
    {"title": "Parasite", "genre": "Thriller", "rating": 8.5, "year": 2019,
     "director": "Bong Joon-ho",
     "description": "A struggling family gradually becomes involved with a wealthy household."},
    {"title": "Get Out", "genre": "Horror", "rating": 7.8, "year": 2017,
     "director": "Jordan Peele",
     "description": "A young man uncovers disturbing secrets during a visit to his girlfriend's family."},
    {"title": "The Conjuring", "genre": "Horror", "rating": 7.5, "year": 2013,
     "director": "James Wan",
     "description": "Paranormal investigators help a family experiencing terrifying supernatural events."},
    {"title": "La La Land", "genre": "Romance", "rating": 8.0, "year": 2016,
     "director": "Damien Chazelle",
     "description": "A musician and actress pursue their dreams while falling in love in Los Angeles."}
]

print(f"Loaded {len(MOVIES)} movies.")

## 2. Movie Tools

In [ ]:
@tool
def search_movies(genre: str) -> str:
    """Search for movies by genre."""
    genre = genre.lower().strip()

    results = [
        movie for movie in MOVIES
        if movie["genre"].lower() == genre
    ]

    if not results:
        return f"No movies found for genre: {genre}"

    return "\n\n".join(
        f"Title: {m['title']}\n"
        f"Genre: {m['genre']}\n"
        f"Rating: {m['rating']}\n"
        f"Year: {m['year']}"
        for m in results
    )

In [ ]:
@tool
def get_movie_details(movie_name: str) -> str:
    """Get detailed information about a specific movie."""
    movie_name = movie_name.lower().strip()

    for movie in MOVIES:
        if movie["title"].lower() == movie_name:
            return (
                f"Title: {movie['title']}\n"
                f"Genre: {movie['genre']}\n"
                f"Rating: {movie['rating']}/10\n"
                f"Year: {movie['year']}\n"
                f"Director: {movie['director']}\n"
                f"Description: {movie['description']}"
            )

    return f"Movie '{movie_name}' was not found." 

In [ ]:
@tool
def top_rated_movies(min_rating: float = 8.0) -> str:
    """Find movies with a rating equal to or higher than min_rating."""
    results = [
        movie for movie in MOVIES
        if movie["rating"] >= min_rating
    ]

    results.sort(key=lambda movie: movie["rating"], reverse=True)

    if not results:
        return f"No movies found with rating >= {min_rating}."

    return "\n".join(
        f"{m['title']} ({m['year']}) - {m['rating']}/10 - {m['genre']}"
        for m in results
    )

In [ ]:
tools = [
    search_movies,
    get_movie_details,
    top_rated_movies
]

print("Available tools:")
for t in tools:
    print("-", t.name)

llm_with_tools = llm.bind_tools(tools)
print("\nTools bound to LLM successfully!")

## 3. LangGraph State

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

## 4. Agent Node

In [ ]:
SYSTEM_PROMPT = """
You are a movie recommendation assistant.

You help users discover movies from the available movie database.

Tools:
1. search_movies - use for genre-based recommendations.
2. get_movie_details - use for a specific movie.
3. top_rated_movies - use for highly rated movies or a rating threshold.

Rules:
- Use tools whenever movie data is required.
- Never invent movies or ratings.
- You may call multiple tools if needed.
- Keep recommendations concise and explain briefly why a movie is worth watching.
"""

def agent_node(state: AgentState):
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

## 5. Build the LangGraph

In [ ]:
tool_node = ToolNode(tools)

workflow = StateGraph(AgentState)

workflow.add_node("agent", agent_node)
workflow.add_node("tools", tool_node)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", tools_condition)
workflow.add_edge("tools", "agent")

graph = workflow.compile()

print("LangGraph compiled successfully!")

In [ ]:
from IPython.display import Image, display

display(
    Image(graph.get_graph().draw_mermaid_png())
)

## 6. Test the Agent

In [ ]:
def run_agent(query: str):
    return graph.invoke({
        "messages": [HumanMessage(content=query)]
    })

result = run_agent("Recommend some action movies.")
print(result["messages"][-1].content)

In [ ]:
result = run_agent("Tell me about Inception.")
print(result["messages"][-1].content)

In [ ]:
result = run_agent("Give me movies rated above 8.5.")
print(result["messages"][-1].content)

In [ ]:
result = run_agent("What makes a movie enjoyable?")
print(result["messages"][-1].content)

## 7. Test Tool Chaining

In [ ]:
result = run_agent(
    "I like action movies. Recommend one highly rated action movie and then give me its details."
)
print(result["messages"][-1].content)

## 8. Inspect the Agent Execution

In [ ]:
def inspect_agent(result):
    for i, message in enumerate(result["messages"]):
        print("\n" + "=" * 60)
        print(f"MESSAGE {i + 1}")
        print("=" * 60)
        print("TYPE:", type(message).__name__)

        if getattr(message, "content", None):
            print("CONTENT:")
            print(message.content)

        if getattr(message, "tool_calls", None):
            print("\nTOOL CALLS:")
            for call in message.tool_calls:
                print("Tool:", call["name"])
                print("Arguments:", call["args"])

result = run_agent("Recommend some action movies.")
inspect_agent(result)

## 9. LangSmith Tracing

In [ ]:
LANGSMITH_API_KEY = userdata.get("LANGSMITH_API_KEY")

if not LANGSMITH_API_KEY:
    print(
        "LANGSMITH_API_KEY not found. "
        "Add it to Colab Secrets to enable tracing."
    )
else:
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = "Movie Recommendation Agent"
    print("LangSmith tracing enabled!")

In [ ]:
# Run this after enabling LangSmith tracing.
result = run_agent("Recommend some highly rated action movies.")
print(result["messages"][-1].content)

## 10. Interactive Movie Agent

In [ ]:
def movie_agent(user_input: str):
    result = graph.invoke({
        "messages": [HumanMessage(content=user_input)]
    })
    return result["messages"][-1].content

print("=" * 60)
print("🎬 MOVIE RECOMMENDATION AGENT")
print("=" * 60)
print("Ask me about movies. Type 'exit' to stop.")
print()

while True:
    user_input = input("You: ")

    if user_input.lower().strip() in ["exit", "quit", "bye"]:
        print("Agent: Goodbye! 🍿")
        break

    try:
        response = movie_agent(user_input)
        print("\nAgent:", response)
        print()
    except Exception as e:
        print("\nError:", str(e))

## Key Concepts

**LangChain**
- `@tool` creates tools.
- `bind_tools()` makes tools available to the LLM.

**LangGraph**
- `StateGraph` defines the workflow.
- `ToolNode` executes the selected tool.
- `tools_condition` routes to the tool or ends the graph.

**LangSmith**
- Traces the LLM calls, tool calls, tool results, and final response.

### Core workflow

**User → Agent → Tool → Agent → Final Answer**
